# Pipeline Robustness Tests

This notebook checks whether the fitted pipeline handles missing values and unseen categories. The data are loaded and split using the same procedure and random seed as in `starter.ipynb`. Tests use modified copies of a reproducible 20-review sample from the held-out test set. They assess successful execution, not prediction accuracy.

## The Data

The dataset has been anonymized and cleaned of missing values. Each row of data represents a review. Eight input features are used to predict `Recommended IND`.

The features can be summarized as follows:

- **Clothing ID**: Categorical identifier, stored as an integer, for the clothing item reviewed.
- **Age**: Positive integer variable of the reviewer's age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Non-negative integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product's high-level division.
- **Department Name**: Categorical name of the product department.
- **Class Name**: Categorical name of the product class.

The target:
- **Recommended IND**: Binary indicator of whether the customer recommends the product: 1 = recommended; 0 = not recommended.

## Load Data

In [1]:
# Load data

import pandas as pd

df = pd.read_csv('data/reviews.csv')

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 18442 entries, 0 to 18441
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing ID              18442 non-null  int64
 1   Age                      18442 non-null  int64
 2   Title                    18442 non-null  str  
 3   Review Text              18442 non-null  str  
 4   Positive Feedback Count  18442 non-null  int64
 5   Division Name            18442 non-null  str  
 6   Department Name          18442 non-null  str  
 7   Class Name               18442 non-null  str  
 8   Recommended IND          18442 non-null  int64
dtypes: int64(4), str(5)
memory usage: 1.3 MB


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses,0
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits,1


## Preparing features (`X`) & target (`y`)

In [2]:
# Separate features from labels

X = df.drop('Recommended IND', axis=1)
y = df['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

Labels: [0 1]
Features:


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits


In [3]:
# Split data into train and test sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    stratify=y,
    random_state=27,
)

## Loading a Sample of the Test Data for Testing

In [4]:
# Load a small sample from the test data for the purpose of testing

import numpy as np

X_test_sample = X_test.sample(n=20, random_state=42)

display(X_test_sample)

,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name
5778,1077,35,"Pretty, dainty, and huge",This dress is very pretty and feminine but was...,0,General Petite,Dresses,Dresses
14979,878,66,Cozy lounger,In the on-line picture this lightweight sweats...,2,General Petite,Tops,Knits
17168,1104,43,Not great,"If you're tall, this dress sits up too high-wa...",18,General Petite,Dresses,Dresses
15655,857,26,Great top!,This top is great. it hangs in the perfect way...,0,General Petite,Tops,Knits
1597,1072,33,Huge,I am between a xs and s but figured i would or...,0,General Petite,Dresses,Dresses
2040,1082,57,Awesome colors and shape,"This dress is gorgeous!! amazing colors, drape...",1,General,Dresses,Dresses
4724,1004,39,Cute skirt,"The fabric was more ""southwestern"" looking whe...",0,General Petite,Bottoms,Skirts
18380,1020,39,"Beautiful, unique piece",Love it! i am 125# and usually a size 2-4 or 2...,0,General Petite,Bottoms,Skirts
10671,828,53,Top is too shear with see through front,This is a sweet top as it is shown. on the mod...,0,General,Tops,Blouses
11802,865,50,Great layer piece,This is a great layering piece. took it on a t...,0,General,Tops,Knits


## Custom Preprocessing Functions

These functions reproduce the custom preprocessing used during training. They must be defined before loading the saved pipeline, which references them when processing test inputs. They are kept unchanged so the tests exercise the same preprocessing as the trained model.

In [5]:
# Create a function to count the number of characters in a review


def review_text_char_count(
    df: pd.DataFrame, col_name: str = 'Review Text'
) -> np.ndarray:

    clean_series = df[col_name].fillna('').astype(str)
    char_counts = clean_series.str.len()

    return char_counts.to_numpy().reshape(-1, 1)

In [6]:
# Test the function

train_df = pd.DataFrame({
    'Review Text': ["Loved every minute of using it.", None, "Great product!"],
})

char_count_matrix = review_text_char_count(train_df, col_name='Review Text')

print("Output Type:", type(char_count_matrix))
print("Output Shape:", char_count_matrix.shape)
print("\n2D Array Output:\n", char_count_matrix)

Output Type: <class 'numpy.ndarray'>
Output Shape: (3, 1)

2D Array Output:
 [[31]
 [ 0]
 [14]]


In [7]:
# Combine Title and Review Text for the text pipeline


def combine_title_and_review(df: pd.DataFrame) -> pd.Series:

    return df['Title'].fillna('') + " " + df['Review Text'].fillna('')

In [8]:
# Tokenize, lemmatize in batches

import spacy

nlp = spacy.load("en_core_web_sm")


def preprocess_reviews_batched(
    reviews: pd.Series, batch_size: int = 64
) -> pd.Series:

    processed_texts = []

    for doc in nlp.pipe(reviews.astype(str), batch_size=batch_size):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_punct and not token.is_space
        ]
        processed_texts.append(" ".join(tokens))

    return pd.Series(processed_texts, index=reviews.index)

## Loading the Fitted Model

In [9]:
# Load the previously fitted model for the purpose of testing

import joblib

model_best = joblib.load('best_model.joblib')

display(model_best)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('randomforestclassifier', ...)]"
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",'pipeline_cache'
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['Clothing ID','Age','Title',...,'Department Name','Class Name','Review']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By speci

In [10]:
# Predict using the sample test data

model_best.predict(X_test_sample)

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1])

## Testing the Pipeline

Each test starts with a fresh copy of the sample and introduces missing values, an unseen category, or both. A test passes if the fitted pipeline returns 20 predictions, each either 0 or 1, without an error. Predictions need not match those for the unchanged sample.

In [11]:
# Helper function for running the robustness tests

def pipeline_robustness(
    df: pd.DataFrame,
    row_pos: int,
    replacement: dict,
) -> np.ndarray:

    df_copy = df.copy()

    for column, value in replacement.items():
        df_copy.loc[df_copy.index[row_pos], column] = value
        print(df_copy[column].iloc[max(0, row_pos - 1):row_pos + 2])

    predictions = model_best.predict(df_copy)

    assert len(predictions) == len(df_copy), "Incorrect prediction count."
    assert np.isin(predictions, [0, 1]).all(), "Unexpected prediction label."

    print("Pass\n")
    return predictions


pipeline_robustness(X_test_sample, row_pos=15, replacement={'Age' : np.nan})
pipeline_robustness(X_test_sample, row_pos=3, replacement={'Department Name' : np.nan})
pipeline_robustness(X_test_sample, row_pos=3, replacement={'Review Text' : np.nan})
pipeline_robustness(X_test_sample, row_pos=3, replacement={'Review Text' : np.nan, 'Title' : np.nan})
pipeline_robustness(X_test_sample, row_pos=3, replacement={'Department Name' : 'Unseen Department'})
pipeline_robustness(X_test_sample, row_pos=3, replacement={
                                                            'Age' : np.nan,
                                                            'Division Name' : np.nan,
                                                            'Department Name' : 'Unseen Department',
                                                            'Title' : np.nan,
                                                            'Review Text' : np.nan
                                                            })

6898     37.0
17461     NaN
510      29.0
Name: Age, dtype: float64
Pass

17168    Dresses
15655        NaN
1597     Dresses
Name: Department Name, dtype: str
Pass

17168    If you're tall, this dress sits up too high-wa...
15655                                                  NaN
1597     I am between a xs and s but figured i would or...
Name: Review Text, dtype: str
Pass

17168    If you're tall, this dress sits up too high-wa...
15655                                                  NaN
1597     I am between a xs and s but figured i would or...
Name: Review Text, dtype: str
17168    Not great
15655          NaN
1597          Huge
Name: Title, dtype: str
Pass

17168              Dresses
15655    Unseen Department
1597               Dresses
Name: Department Name, dtype: str
Pass

17168    43.0
15655     NaN
1597     33.0
Name: Age, dtype: float64
17168    General Petite
15655               NaN
1597     General Petite
Name: Division Name, dtype: str
17168              Dresses
15655   

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1])

The fitted pipeline successfully generated predictions for a 20-review sample containing missing numerical, categorical, and text values, an unseen category, and combined conditions. These checks demonstrate successful handling of the tested inputs; they do not establish prediction accuracy when information is missing.